# 20260725_EDA_구조환경지표_통합파일_결측치_검증
- 작성자: 이정연
- 이슈 #38 참고


## 1. 실행 환경 설정


In [1]:
import sys
from pathlib import Path

import pandas as pd

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / ".git").exists()), None
)
if repo_root is None:
    raise FileNotFoundError("현재 실행 위치의 상위 경로에서 Git 저장소를 찾지 못했습니다.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.evaluation.structural_validation import (  # noqa: E402
    compare_region_year_matrices,
    to_numeric_strict,
    to_verification_record,
)

print(f"저장소 루트: {repo_root}")

저장소 루트: /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha


## 1-1. 공통 설정 - 지역명 매핑

`20260724_EDA_구조환경지표_21개_원자료기반_전수검증.ipynb`에서 쓴 것과 동일한 방식으로,
git 추적 대상인 `data/lookup/시도_지역코드_매핑.csv`를 지역명 단일 소스로 사용한다.


In [2]:
lookup_path = repo_root / "data" / "lookup" / "시도_지역코드_매핑.csv"
region_lookup = pd.read_csv(lookup_path)

region_order = ["전국", *region_lookup["지역"].tolist()]

LEGACY_REGION_NAMES = {
    "강원도": "강원",
    "전라북도": "전북",
    "제주도": "제주",
}
NATIONWIDE_ALIASES = {"전체": "전국", "계": "전국"}

region_map = (
    {r: r for r in region_order}
    | dict(zip(region_lookup["지역명_전체"], region_lookup["지역"]))
    | LEGACY_REGION_NAMES
    | NATIONWIDE_ALIASES
)

print(f"region_order: {len(region_order)}개 지역 (룩업 {len(region_lookup)}개 시도 + 전국)")

region_order: 18개 지역 (룩업 17개 시도 + 전국)


## 1-2. 공통 설정 - 검증 결과 누적


In [3]:
verification_records: list[dict[str, object]] = []
years_all = list(range(2016, 2026))
measures_dir = repo_root / "data" / "raw" / "지표별_측정값"

## 1-3. 공통 설정 - 명세(YAML) 로드

가공 파일 이름은 이미 `configs/structural_indicators_verification.yaml`에 있으므로, 여기서 다시
타이핑하지 않고 그대로 읽어와 재사용한다.


In [4]:
import yaml

config_path = repo_root / "configs" / "structural_indicators_verification.yaml"
with config_path.open(encoding="utf-8") as file:
    spec_config = yaml.safe_load(file)

file_name_by_id = {
    item["id"]: item["finance_team_value"]["file_name"] for item in spec_config["indicators"]
}
print(f"명세: {config_path.relative_to(repo_root)} | 지표 {len(file_name_by_id)}개")

명세: configs/structural_indicators_verification.yaml | 지표 21개


## 2. 통합 파일(합본) 로드

이슈 #38 작업 목록의 "통합 파일(3번)의 merge 결과가 개별 가공 파일 값과 일치하는지 검증"에 해당하는 작업이다.
재정팀이 21개 지표를 지역×연도로 합쳐 만든 `구조환경지표 측정값_합본_30개 중 21개.xlsx`를 그대로 로드하고,
지표별로 나눠서 각 지표의 원래 가공 파일(개별 xlsx)에 있는 결과 시트와 대조한다.


In [5]:
merged_path = measures_dir / "구조환경지표 측정값_합본_30개 중 21개.xlsx"
merged_df = pd.read_excel(merged_path, sheet_name="Sheet1")
merged_df.columns = [c if isinstance(c, int) else str(c).strip() for c in merged_df.columns]

print(f"통합 파일: {merged_path.relative_to(repo_root)}")
print(f"크기: {merged_df.shape[0]}행 x {merged_df.shape[1]}열")
print(f"세부지표 종류: {merged_df['세부지표'].nunique()}개")
print(f"지역 표기: {sorted(merged_df['지역'].unique())}")

통합 파일: data/raw/지표별_측정값/구조환경지표 측정값_합본_30개 중 21개.xlsx
크기: 377행 x 15열
세부지표 종류: 21개
지역 표기: ['강원', '경기', '경남', '경북', '광주', '대구', '대전', '부산', '서울', '세종', '울산', '인천', '전국', '전남', '전북', '전체', '제주', '충남', '충북']


## 3. 지표별 merge 검증

21개 지표 각각에 대해 (개별 가공 파일의 결과 시트 → 통합 파일에서 해당 세부지표만 뽑은 슬라이스)를 대조한다.
지표·파일명·순서는 전부 `configs/structural_indicators_verification.yaml`에서 그대로 가져온다.

결과 시트는 모든 파일에서 항상 첫 번째 시트임을 확인했다(파일 안 시트 이름이 프로젝트 코드와 다르게
붙어 있는 경우가 있어 - 예: 산후조리원 이용 요금 파일의 첫 시트 이름이 "3-2.1. 산후조리원 보급도"로
잘못 붙어 있음 - 이름 대신 위치(첫 번째 시트)로 접근한다).

통합 파일의 "세부지표" 라벨은 대부분 명세의 `name`과 같지만, 간격·괄호 표기가 달라 일치하지 않는
5개 지표만 `LABEL_OVERRIDES`로 예외 처리한다.


In [6]:
# 통합 파일의 "세부지표" 라벨이 명세(YAML)의 name과 다른 5개 지표만 예외로 남긴다.
LABEL_OVERRIDES = {
    "income_level": "소득수준",
    "delivery_bed_supply": "분만실 병상수 보급도",
    "pediatric_specialist_supply": "소아청소년과 전문인력 보급도",
    "family_friendly_certification_rate": "가족친화인증기업 비율",
    "housework_gender_equality": "가사 분담에 대한 성평등 인식",
}

indicator_names = {
    item["id"]: LABEL_OVERRIDES.get(item["id"], item["name"]) for item in spec_config["indicators"]
}
print(f"라벨 예외 {len(LABEL_OVERRIDES)}건, 전체 지표 {len(indicator_names)}개")

라벨 예외 5건, 전체 지표 21개


In [7]:
def _clean_region_year_table(df: pd.DataFrame) -> pd.DataFrame:
    """지역을 인덱스로, 2016~2025 연도를 컬럼으로 갖는 표로 정리한다."""
    df = df.copy()
    df["지역"] = df["지역"].astype(str).str.strip().map(lambda r: region_map.get(r, r))
    df = df.set_index("지역")
    df = df[~df.index.duplicated(keep="first")]
    df = df.loc[df.index.isin(region_order)]
    year_cols = [c for c in years_all if c in df.columns]
    for col in year_cols:
        df[col] = to_numeric_strict(df[col])
    return df.reindex(columns=years_all)


def load_result_sheet(file_name: str) -> pd.DataFrame:
    """개별 가공 파일의 결과 시트(항상 첫 번째 시트)를 지역×연도 표로 읽는다."""
    df = pd.read_excel(measures_dir / file_name, sheet_name=0)
    df.columns = [c if isinstance(c, int) else str(c).strip() for c in df.columns]
    return _clean_region_year_table(df)


def load_merged_slice(label: str) -> pd.DataFrame:
    """통합 파일에서 세부지표 라벨이 일치하는 행만 뽑아 지역×연도 표로 만든다."""
    sub = merged_df[merged_df["세부지표"] == label]
    if sub.empty:
        raise KeyError(f"통합 파일에서 세부지표 '{label}'을 찾지 못했습니다.")
    return _clean_region_year_table(sub)


merge_check_results: dict[str, object] = {}
for item in spec_config["indicators"]:
    indicator_id = item["id"]
    result_df = load_result_sheet(item["finance_team_value"]["file_name"])
    merged_slice = load_merged_slice(indicator_names[indicator_id])

    expected_regions = (
        [r for r in region_order if r != "전국"] if indicator_id == "work_hours" else region_order
    )

    check = compare_region_year_matrices(
        merged_slice,
        result_df,
        expected_regions=expected_regions,
        expected_years=years_all,
        tolerance=1e-6,
        label=f"{indicator_id} - 통합 파일 vs 가공 파일 결과 대조",
    )
    merge_check_results[indicator_id] = check
    verification_records.append(
        to_verification_record(check, indicator_id=indicator_id, stage="통합 파일 merge 대조")
    )

merge_summary = pd.DataFrame(verification_records)
display(merge_summary)
print("\n판정별 건수:")
print(merge_summary["판정"].value_counts())

,지표ID,검증단계,비교건수,최대절대오차,불일치건수,결측조합수,최대오차위치,판정
0,youth_employment_rate,통합 파일 merge 대조,178,0.0,0,2,"(전국, 2016)",확인 필요
1,work_hours,통합 파일 merge 대조,166,0.0,0,4,"(서울, 2016)",확인 필요
2,income_satisfaction,통합 파일 merge 대조,90,0.0,0,90,"(전국, 2017)",확인 필요
3,income_level,통합 파일 merge 대조,162,0.0,0,18,"(전국, 2016)",확인 필요
4,childcare_capacity_rate,통합 파일 merge 대조,180,0.0,0,0,"(전국, 2016)",정상
5,after_school_care_supply,통합 파일 merge 대조,162,0.0,0,18,"(전국, 2016)",확인 필요
6,private_education_cost,통합 파일 merge 대조,180,0.0,0,0,"(전국, 2016)",정상
7,cultural_facilities_supply,통합 파일 merge 대조,162,0.0,0,18,"(전국, 2016)",확인 필요
8,urban_park_supply,통합 파일 merge 대조,162,0.0,0,18,"(전국, 2016)",확인 필요
9,leisure_satisfaction,통합 파일 merge 대조,90,0.0,0,90,"(전국, 2017)",확인 필요



판정별 건수:
판정
확인 필요    17
정상        4
Name: count, dtype: int64


불일치가 발견된 지표가 있으면 상세를 확인한다(오차 0이 아니거나 결측 조합이 있는 경우).


In [8]:
for indicator_id, check in merge_check_results.items():
    if check.mismatch_count > 0:
        print(f"=== {indicator_id}: 불일치 {check.mismatch_count}건 ===")
        display(check.detail)

## 4. 결측치 현황 전수 확인

통합 파일 기준으로 지표별 결측 현황을 센다. 한 연도에 **모든 지역이 결측**이면 그 지표가 해당 연도를
아예 조사/공표하지 않는 구간(원자료 자체 결측)일 가능성이 높고, **일부 지역만 결측**이면 원자료는
있는데 가공 과정에서 빠졌을 가능성(가공 과정 누락 후보)이 있다는 뜻이라 이 둘을 구분해서 집계한다.


In [9]:
missing_rows = []
for indicator_id, label in indicator_names.items():
    merged_slice = load_merged_slice(label)
    n_regions = len(merged_slice.index)

    for year in years_all:
        if year not in merged_slice.columns:
            continue
        col = merged_slice[year]
        n_missing = int(col.isna().sum())
        if n_missing == 0:
            continue
        kind = (
            "원자료 자체 결측(연도 전체)"
            if n_missing == n_regions
            else "가공 과정 누락 후보(지역 일부만)"
        )
        missing_rows.append(
            {
                "지표ID": indicator_id,
                "연도": year,
                "결측_지역수": n_missing,
                "전체_지역수": n_regions,
                "구분": kind,
            }
        )

missing_summary = pd.DataFrame(missing_rows)
display(missing_summary)

print("\n구분별 건수:")
print(missing_summary["구분"].value_counts())

print("\n가공 과정 누락 후보(지역 일부만 결측)만 모아보기:")
display(missing_summary[missing_summary["구분"] == "가공 과정 누락 후보(지역 일부만)"])
print(
    '\n참고: 이 방식은 "연도 전체 결측=원자료 결측, 지역 일부만 결측=가공 누락"이라는 '
    "휴리스틱이라, 특정 지역만 원자료 자체가 없는 경우(세종·충남 등 신설/표본 부족 지자체)도 "
    '"가공 과정 누락 후보"로 잡힌다. 위에 나온 5건은 실제로 20260724 노트북에서 이미 확인한 '
    "정상 결측이다(청년고용률 2016년 세종·충남, 근로시간 2016-2019년 세종 - 모두 원자료 자체 결측 "
    "으로 문서화됨). 즉 이번 통합 파일 검증에서 새로 발견된 미상 결측은 없다."
)

,지표ID,연도,결측_지역수,전체_지역수,구분
0,youth_employment_rate,2016,2,18,가공 과정 누락 후보(지역 일부만)
1,work_hours,2016,1,17,가공 과정 누락 후보(지역 일부만)
2,work_hours,2017,1,17,가공 과정 누락 후보(지역 일부만)
3,work_hours,2018,1,17,가공 과정 누락 후보(지역 일부만)
4,work_hours,2019,1,17,가공 과정 누락 후보(지역 일부만)
...,...,...,...,...,...
58,housework_gender_equality,2017,18,18,원자료 자체 결측(연도 전체)
59,housework_gender_equality,2019,18,18,원자료 자체 결측(연도 전체)
60,housework_gender_equality,2021,18,18,원자료 자체 결측(연도 전체)
61,housework_gender_equality,2023,18,18,원자료 자체 결측(연도 전체)



구분별 건수:
구분
원자료 자체 결측(연도 전체)       58
가공 과정 누락 후보(지역 일부만)     5
Name: count, dtype: int64

가공 과정 누락 후보(지역 일부만 결측)만 모아보기:


,지표ID,연도,결측_지역수,전체_지역수,구분
0,youth_employment_rate,2016,2,18,가공 과정 누락 후보(지역 일부만)
1,work_hours,2016,1,17,가공 과정 누락 후보(지역 일부만)
2,work_hours,2017,1,17,가공 과정 누락 후보(지역 일부만)
3,work_hours,2018,1,17,가공 과정 누락 후보(지역 일부만)
4,work_hours,2019,1,17,가공 과정 누락 후보(지역 일부만)



참고: 이 방식은 "연도 전체 결측=원자료 결측, 지역 일부만 결측=가공 누락"이라는 휴리스틱이라, 특정 지역만 원자료 자체가 없는 경우(세종·충남 등 신설/표본 부족 지자체)도 "가공 과정 누락 후보"로 잡힌다. 위에 나온 5건은 실제로 20260724 노트북에서 이미 확인한 정상 결측이다(청년고용률 2016년 세종·충남, 근로시간 2016-2019년 세종 - 모두 원자료 자체 결측 으로 문서화됨). 즉 이번 통합 파일 검증에서 새로 발견된 미상 결측은 없다.


## 5. 검증 결과 요약


In [10]:
verification_summary = pd.DataFrame(verification_records)
verification_summary.insert(1, "지표명", verification_summary["지표ID"].map(indicator_names))
display(verification_summary)

print("\n판정별 건수:")
print(verification_summary["판정"].value_counts())

,지표ID,지표명,검증단계,비교건수,최대절대오차,불일치건수,결측조합수,최대오차위치,판정
0,youth_employment_rate,청년고용률,통합 파일 merge 대조,178,0.0,0,2,"(전국, 2016)",확인 필요
1,work_hours,근로시간,통합 파일 merge 대조,166,0.0,0,4,"(서울, 2016)",확인 필요
2,income_satisfaction,소득만족도,통합 파일 merge 대조,90,0.0,0,90,"(전국, 2017)",확인 필요
3,income_level,소득수준,통합 파일 merge 대조,162,0.0,0,18,"(전국, 2016)",확인 필요
4,childcare_capacity_rate,보육시설 보급률,통합 파일 merge 대조,180,0.0,0,0,"(전국, 2016)",정상
5,after_school_care_supply,방과후 돌봄시설 보급도,통합 파일 merge 대조,162,0.0,0,18,"(전국, 2016)",확인 필요
6,private_education_cost,사교육비 지출액,통합 파일 merge 대조,180,0.0,0,0,"(전국, 2016)",정상
7,cultural_facilities_supply,문화기반시설 보급도,통합 파일 merge 대조,162,0.0,0,18,"(전국, 2016)",확인 필요
8,urban_park_supply,도시공원 보급도,통합 파일 merge 대조,162,0.0,0,18,"(전국, 2016)",확인 필요
9,leisure_satisfaction,여가생활 만족도,통합 파일 merge 대조,90,0.0,0,90,"(전국, 2017)",확인 필요



판정별 건수:
판정
확인 필요    17
정상        4
Name: count, dtype: int64
